In [1]:
from google.colab import drive
# drive.mount('/content/gdrive')
drive.mount("/content/gdrive", force_remount=True)

Mounted at /content/gdrive


In [0]:
import os
import cv2
import pandas as pd

In [3]:
current_path = os.chdir("/content/gdrive/My Drive/CV_final_assignment")
current_path = os.getcwd()
current_path

'/content/gdrive/My Drive/CV_final_assignment'

In [4]:
!pip install numpy==1.16.1
import numpy as np

In [0]:
x_train = np.load('./x_seg_train_.npy') 
x_val = np.load("./x_seg_val.npy")
x_test = np.load("./x_seg_test.npy")

y_train = np.load('./y_seg_train_.npy') 
y_val = np.load("./y_seg_val.npy")
y_test = np.load("./y_seg_test.npy")

In [6]:
from keras.utils import to_categorical
y_train = to_categorical(y_train)
y_val = to_categorical(y_val)

Using TensorFlow backend.


In [7]:
y_train.shape

(1049, 256, 256, 21)

In [8]:
!pip install tensorflow_gpu==2.0

     |████████████████████████████████| 380.8MB 49kB/s 
     |████████████████████████████████| 450kB 32.6MB/s 
     |████████████████████████████████| 3.8MB 29.7MB/s 
  Created wheel for gast: filename=gast-0.2.2-cp36-none-any.whl size=7540 sha256=27988122458383a530490a84ba222f3b5553d7aee7e45228c12cd115da76c490
  Stored in directory: /root/.cache/pip/wheels/5c/2e/7e/a1d4d4fcebe6c381f378ce7743a3ced3699feb89bcfbdadadd
Successfully built gast
ERROR: tensorflow 2.2.0rc4 has requirement gast==0.3.3, but you'll have gast 0.2.2 which is incompatible.
ERROR: tensorflow 2.2.0rc4 has requirement tensorboard<2.3.0,>=2.2.0, but you'll have tensorboard 2.0.2 which is incompatible.
ERROR: tensorflow 2.2.0rc4 has requirement tensorflow-estimator<2.3.0,>=2.2.0, but you'll have tensorflow-estimator 2.0.1 which is incompatible.
ERROR: tensorflow-probability 0.10.0rc0 has requirement gast>=0.3.2, but you'll have gast 0.2.2 which is incompatible.
  Found existing installation: tensorflow-estimator 2.2.0


In [0]:
from keras.preprocessing.image import load_img
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input
from keras.applications.vgg16 import decode_predictions
from keras.applications.vgg16 import VGG16
from keras.layers import Flatten, Dense,Conv2D,Dropout,UpSampling2D, Input,Conv2DTranspose,MaxPooling2D
from keras.models import Model
from keras.layers.normalization import BatchNormalization
from keras.layers import concatenate
from keras.optimizers import Adam,RMSprop
import tensorflow as tf
from keras import backend as K
from keras.losses import categorical_crossentropy


In [0]:
#%% Load data
batch_size = 32
num_classes = 20    # this is the actual number of classes, void and backgournd are added later

# train_gen = train_data_gen('train_segm.txt', batch_size, num_classes)
# val_gen = val_data_gen('val_segm.txt', batch_size, num_classes)
#test_gen = val_data_gen('segm_test.txt', batch_size, num_classes)

print('LOADING DATA FINISHED')

#%% Define neural network
img_input = Input(shape=(256,256,3))
### AANPASSEN kernel_initializer = 'he_normal'
"""
enc=conv drop conv pool
5=conv drop conv
dec=convtrafo cat conv drop conv
out=conv sigmoid
"""
# Block 1
conv1 = Conv2D(64, (3,3), activation='relu', padding='same', kernel_initializer = 'he_normal')(img_input)
drop1 = Dropout(0.2)(conv1)
conv1 = Conv2D(64, (3,3), activation='relu', padding='same', kernel_initializer = 'he_normal')(drop1)
pool1 = MaxPooling2D((2,2), strides=(2,2), name='block1_pool')(conv1)
# Block 2
conv2 = Conv2D(128, (3,3), activation='relu', padding='same', kernel_initializer = 'he_normal')(pool1)
drop2 = Dropout(0.2)(conv2)
conv2 = Conv2D(128, (3,3), activation='relu', padding='same', kernel_initializer = 'he_normal')(drop2)
pool2 = MaxPooling2D((2,2), strides=(2,2), name='block2_pool')(conv2)
# Block 3
conv3 = Conv2D(256, (3,3), activation='relu', padding='same', kernel_initializer = 'he_normal')(pool2)
drop3 = Dropout(0.2)(conv3)
conv3 = Conv2D(256, (3,3), activation='relu', padding='same', kernel_initializer = 'he_normal')(drop3)
pool3 = MaxPooling2D((2,2), strides=(2,2), name='block3_pool')(conv3)
# Block 4
conv4 = Conv2D(512, (3,3), activation='relu', padding='same', kernel_initializer = 'he_normal')(pool3)
drop4 = Dropout(0.2)(conv4)
conv4 = Conv2D(512, (3,3), activation='relu', padding='same', kernel_initializer = 'he_normal')(drop4)
pool4 = MaxPooling2D((2,2), strides=(2,2), name='block4_pool')(conv4)

# Block 5
conv5 = Conv2D(1024, (3,3), activation='relu', padding='same', kernel_initializer = 'he_normal')(pool4)
drop5 = Dropout(0.2)(conv5)
conv5 = Conv2D(1024, (3,3), activation='relu', padding='same', kernel_initializer = 'he_normal')(drop5)

# Block 6
up6 = Conv2DTranspose(512, (3,3), strides=(2,2), activation='relu', padding='same',  kernel_initializer = 'he_normal')(conv5)
cat6 = concatenate([up6, conv4], axis=-1)
conv6 = Conv2D(512, (3,3), activation='relu', padding='same',  kernel_initializer = 'he_normal')(cat6)
drop6 = Dropout(0.2)(conv6)
conv6 = Conv2D(512, (3,3), activation='relu', padding='same',  kernel_initializer = 'he_normal')(drop6)
# Block 7
up7 = Conv2DTranspose(256, (3,3), strides=(2,2), activation='relu', padding='same',  kernel_initializer = 'he_normal')(conv6)
cat7 = concatenate([up7, conv3], axis=-1)
conv7 = Conv2D(256, (3,3), activation='relu', padding='same',  kernel_initializer = 'he_normal')(cat7)
drop7 = Dropout(0.2)(conv7)
conv7 = Conv2D(256, (3,3), activation='relu', padding='same',  kernel_initializer = 'he_normal')(drop7)
# Block 8
up8 = Conv2DTranspose(128, (3,3), strides=(2,2), activation='relu', padding='same',  kernel_initializer = 'he_normal')(conv7)
cat8 = concatenate([up8, conv2], axis=-1)
conv8 = Conv2D(128, (3,3), activation='relu', padding='same',  kernel_initializer = 'he_normal')(cat8)
drop8 = Dropout(0.2)(conv8)
conv8 = Conv2D(128, (3,3), activation='relu', padding='same',  kernel_initializer = 'he_normal')(drop8)
# Block 9
up9 = Conv2DTranspose(64, (3,3), strides=(2,2), activation='relu', padding='same')(conv8)
cat9 = concatenate([up9, conv1], axis=-1)
conv9 = Conv2D(64, (3,3), activation='relu', padding='same',  kernel_initializer = 'he_normal')(cat9)
drop9 = Dropout(0.2)(conv9)
conv9 = Conv2D(64, (3,3), activation='relu', padding='same',  kernel_initializer = 'he_normal')(drop9)

out = Conv2D(21, (1,1), activation='sigmoid', padding='same',  kernel_initializer = 'he_normal')(conv9)

#%% Compile neural network
model = Model(img_input, out) # this would build the segmentation model

model.compile(loss='categorical_crossentropy',
              optimizer=RMSprop(lr=1e-4),
              metrics=['acc'])

print('COMPILING MODEL FINISHED')
#%% Train neural network
current_path = os.getcwd()
weights_path  = os.path.join(r'C:\Users\astri\Documenten\__5e jaar\Vision\Project', 'segmentation')

# Create of list of callbacks for intermediate results
# model_names = weights_path + '\weights.{epoch:02d}-{val_acc:.2f}.hdf5'
# checkpoint = ModelCheckpoint(model_names, monitor='val_acc', 
#                               verbose=1, save_best_only=True, mode='max')

# #os.makedirs(weights_path+'\log.out', mode=0o777, exist_ok=True)
# csv_logger = CSVLogger(weights_path+'\log.out', append=True, separator=';')

# earlystopping = EarlyStopping(monitor='val_acc', verbose=1,
#                               min_delta=0.01, patience=10, mode='max')
# callbacks_list = [checkpoint, csv_logger, earlystopping]

# # Train model using training and validation generators
# results = model.fit_generator(train_gen,
#                           epochs=10, 
#                           steps_per_epoch=24,
#                           validation_data=val_gen, 
#                           validation_steps=24, 
#                           callbacks=callbacks_list)

model.fit(x=x_train,y=y_train,batch_size=16,epochs=20,validation_data=(x_val,y_val))

LOADING DATA FINISHED
COMPILING MODEL FINISHED
Train on 1049 samples, validate on 301 samples
Epoch 1/20
1049/1049 [==============================] - 180s 172ms/step - loss: 1.9878 - acc: 0.7483 - val_loss: 1.5076 - val_acc: 0.7886
Epoch 2/20
1049/1049 [==============================] - 143s 137ms/step - loss: 1.3524 - acc: 0.8095 - val_loss: 1.3153 - val_acc: 0.7886
Epoch 3/20
1049/1049 [==============================] - 143s 136ms/step - loss: 1.2448 - acc: 0.8097 - val_loss: 1.4235 - val_acc: 0.7886
Epoch 4/20
1049/1049 [==============================] - 143s 136ms/step - loss: 1.1765 - acc: 0.8097 - val_loss: 1.4172 - val_acc: 0.7886
Epoch 5/20
1049/1049 [==============================] - 143s 136ms/step - loss: 1.1367 - acc: 0.8097 - val_loss: 1.1490 - val_acc: 0.7886
Epoch 6/20
1049/1049 [==============================] - 143s 136ms/step - loss: 1.0876 - acc: 0.8097 - val_loss: 1.3662 - val_acc: 0.7886
Epoch 7/20
1049/1049 [==============================] - 143s 136ms/step - loss